# 04 — Customer Segmentation (RFM + K-Means)

Two segmentations of the customer base, then a comparison:

1. **Rule-based RFM** (this notebook, Day 19) — score each customer 1–5 on Recency,
   Frequency and Monetary value, then map to business labels.
2. **K-Means clustering** (Day 20) — let an algorithm group similar customers, and
   compare it to the hand-built rules.

Input is `customer_rfm.parquet` from notebook 02 — one row per customer, already
holding recency (days since last order), frequency (number of orders) and monetary
(lifetime revenue).

In [1]:
import pandas as pd

rfm = pd.read_parquet("../data/processed/customer_rfm.parquet")
rfm.shape

(5852, 4)

In [2]:
rfm[["recency", "frequency", "monetary"]].describe().round(1)

,recency,frequency,monetary
count,5852.0,5852.0,5852.0
mean,200.2,6.3,2916.7
std,208.5,12.7,14306.9
min,1.0,1.0,3.0
25%,25.0,1.0,339.6
50%,95.0,3.0,856.0
75%,379.0,7.0,2241.0
max,739.0,373.0,580987.0


## Rule-based RFM scoring

Score each customer 1–5 on each dimension using quintiles (`pd.qcut` splits the
customers into five equal-sized groups).

- **Recency** is reverse-scored: a *low* number of days means a recent buyer, so it
  earns a *high* score (labels `[5,4,3,2,1]`).
- **Frequency** and **Monetary** score normally: higher = better (`[1,2,3,4,5]`).

Frequency needs `.rank(method="first")` before `qcut`. 1,618 customers have exactly
one order, so the raw quintile edges collide (both the 20th and 40th percentile are
"1 order") and `qcut` refuses to build bins. Ranking first breaks those ties so the
five groups come out equal.

In [3]:
rfm["R"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm[["recency", "R", "frequency", "F", "monetary", "M"]].head()

,recency,R,frequency,F,monetary,M
0,326,2,3,3,77352.96,5
1,2,5,8,4,4921.53,5
2,75,3,5,4,1658.40,4
3,19,5,3,3,3678.69,5
4,310,2,1,1,294.40,2


## Map scores to business segments

A simple, mutually-exclusive rule on the Recency and Frequency scores. The rules are
checked top to bottom, so each customer lands in exactly one segment:

| Segment | Rule | Meaning |
|---|---|---|
| **Champions** | R≥4 and F≥4 | recent *and* frequent — the best customers |
| **Loyal** | R≥3 and F≥3 | solidly active |
| **Potential** | R≥3 and F<3 | recent but not yet frequent — room to grow |
| **At Risk** | R<3 and F≥3 | used to buy often, now gone quiet — the ones to win back |
| **Lost** | otherwise | low on both |

Recency and Frequency carry the segmentation; Monetary is kept for profiling the
segments afterwards.

In [4]:
def to_segment(row):
    r, f = row["R"], row["F"]
    if r >= 4 and f >= 4:
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal"
    if r >= 3 and f < 3:
        return "Potential"
    if r < 3 and f >= 3:
        return "At Risk"
    return "Lost"

rfm["segment"] = rfm.apply(to_segment, axis=1)

segment_order = ["Champions", "Loyal", "Potential", "At Risk", "Lost"]
rfm["segment"].value_counts().reindex(segment_order)

segment
Champions    1476
Loyal        1207
Potential     833
At Risk       828
Lost         1508
Name: count, dtype: int64

In [5]:
segment_profile = rfm.groupby("segment").agg(
    customers=("segment", "size"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum"),
).reindex(segment_order).round(1)

segment_profile

,customers,avg_recency,avg_frequency,avg_monetary,total_revenue
segment,,,,,
Champions,1476,20.4,15.6,8005.7,11816435.5
Loyal,1207,78.2,5.4,2006.2,2421467.1
Potential,833,64.1,1.4,741.2,617410.3
At Risk,828,368.0,5.0,1923.9,1592996.0
Lost,1508,456.9,1.2,411.3,620259.0


In [6]:
total_revenue = rfm["monetary"].sum()
revenue_share = (rfm.groupby("segment")["monetary"].sum() / total_revenue * 100)

revenue_share.reindex(segment_order).round(1)

segment
Champions    69.2
Loyal        14.2
Potential     3.6
At Risk       9.3
Lost          3.6
Name: monetary, dtype: float64

**Takeaway.** The base splits cleanly into five segments that behave very differently:

- **Champions (1,476 customers) generate 69% of all revenue** — recent, frequent, ~£8,000
  lifetime value each. Losing even a few is expensive; these are the accounts to protect first.
- **At Risk (828 customers) hold £1.59M — 9.3% of revenue.** They averaged 5 orders and ~£1,900
  each, but haven't purchased in ~368 days. This is money actively walking out the door and the
  single most actionable group: they have proven value and a clear re-engagement trigger.
- **Lost (1,508) and Potential (833)** are low-value today — Lost are gone, Potential are new or
  occasional and worth nurturing toward Loyal.

**Recommendation seed for Day 27:** the **£1.59M in the At Risk segment** is the concrete number to
attach a win-back campaign to — it quantifies exactly how much revenue a retention programme is
defending, which is the whole business case for retention spend.